# Strategy research template

Use this notebook to explore a single strategy idea before encoding it in Freqtrade.

Typical flow:
1. Load OHLCV data for one or more pairs.
2. Compute indicators.
3. Define entry/exit rules and compute a backtest summary (Sharpe, drawdown, win rate).
4. Plot the equity curve and inspect trades.
5. Once promising, port the rules to `user_data/strategies/<YourStrategy>.py` and run
   `freqtrade backtesting` for a proper walk-forward + fee/slippage simulation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import talib.abstract as ta

DATA_DIR = Path('../data/okx')  # adjust to wherever your data lives
TIMEFRAME = '5m'
PAIR = 'BTC/USDT'

filename = DATA_DIR / f"{PAIR.replace('/', '_')}-{TIMEFRAME}.feather"
df = pd.read_feather(filename)
df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
df = df.set_index('date').astype(float)
df.head()

In [ ]:
# Example indicators.
df['ema_fast'] = ta.EMA(df, timeperiod=9)
df['ema_slow'] = ta.EMA(df, timeperiod=21)
df['ema_trend'] = ta.EMA(df, timeperiod=100)
df.tail()

In [ ]:
# Vectorised toy backtest: long when fast > slow and above trend, flat otherwise.
signal = ((df['ema_fast'] > df['ema_slow']) & (df['close'] > df['ema_trend'])).astype(int)
position = signal.shift(1).fillna(0)
ret = df['close'].pct_change().fillna(0)
strat_ret = position * ret

equity = (1 + strat_ret).cumprod()
buy_hold = (1 + ret).cumprod()

sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(365 * 24 * 12) if strat_ret.std() else 0.0
max_dd = (equity / equity.cummax() - 1).min()

print(f"Strategy total return: {(equity.iloc[-1] - 1) * 100:.2f}%")
print(f"Buy & hold total return: {(buy_hold.iloc[-1] - 1) * 100:.2f}%")
print(f"Annualised Sharpe (no fees): {sharpe:.2f}")
print(f"Max drawdown: {max_dd * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
equity.plot(ax=ax, label='strategy')
buy_hold.plot(ax=ax, label='buy & hold', alpha=0.7)
ax.set_title(f'{PAIR} {TIMEFRAME} — toy EMA-trend strategy vs buy & hold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Important caveats

- **No transaction costs / slippage**: real trades lose 5–20 bps per round trip on majors.
- **No fills modelling**: this assumes you trade exactly at the close, which you won't.
- **Single in-sample fit**: an indicator that looks great here will likely break out of sample.
- **Survivorship**: pairs that didn't survive (delisted alts) aren't here.

Treat any result here as a *signal that something is worth a proper Freqtrade backtest*, not as proof that the strategy works.